In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import joblib
from sklearn.preprocessing import LabelEncoder


# 1. Cargar los datos
df = pd.read_csv('../data/consolidado_transformado.csv')

# Extraer solo la fecha (sin la hora) para poder agrupar por día
df['fecha_hora'] = pd.to_datetime(df['fecha_hora'])
df['fecha_sola'] = df['fecha_hora'].dt.date

# 2. FEATURE ENGINEERING: Calcular "Volumen de compra del día por bodega"
# Agrupamos por fecha y bodega y contamos cuántas transacciones hubo
volumen_diario = df.groupby(['fecha_sola', 'id_bodega'])['id_transaccion'].count().reset_index()
volumen_diario.rename(columns={'id_transaccion': 'volumen_compra_dia'}, inplace=True)

# Unimos esta nueva variable a nuestro dataset principal
df = df.merge(volumen_diario, on=['fecha_sola', 'id_bodega'], how='left')

# 3. Seleccionar variables
variables_entrada = ['pais', 'id_bodega', 'categoria', 'volumen_compra_dia']
variable_objetivo = 'dias_demora_real'

df_modelo = df[variables_entrada + [variable_objetivo]].dropna().copy()

# 4. OPTIMIZACIÓN: Codificación híbrida para evitar explosión de memoria
# A. Label Encoding para bodegas (pasa de 2000 columnas a solo 1)
le_bodegas = LabelEncoder()
df_modelo['id_bodega_encoded'] = le_bodegas.fit_transform(df_modelo['id_bodega'].astype(str))

# Definimos nuestras nuevas variables X (usando la bodega codificada)
X = df_modelo[['pais', 'categoria', 'volumen_compra_dia', 'id_bodega_encoded']]
y = df_modelo[variable_objetivo]

# B. One-Hot Encoding solo para país y categoría (generará muy pocas columnas)
X_encoded = pd.get_dummies(X, columns=['pais', 'categoria'], drop_first=True)

# 5. División en Train/Test (80/20)
X_train, X_test, y_train, y_test = train_test_split(X_encoded, y, test_size=0.2, random_state=42)

# 6. Entrenamiento del Modelo
print("Entrenando Random Forest (versión optimizada)...")
modelo_regresion = RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1)
modelo_regresion.fit(X_train, y_train)
print("¡Entrenamiento finalizado en segundos!")

# 7. Evaluación del Modelo
predicciones = modelo_regresion.predict(X_test)

mae = mean_absolute_error(y_test, predicciones)
rmse = np.sqrt(mean_squared_error(y_test, predicciones))
r2 = r2_score(y_test, predicciones)

print("\n--- MÉTRICAS DEL MODELO ---")
print(f"MAE (Error Absoluto Medio): {mae:.2f} días")
print(f"RMSE (Raíz Error Cuadrático): {rmse:.2f} días")
print(f"R² (Coeficiente de Determinación): {r2:.4f}")

# 8. Guardar todo
joblib.dump(modelo_regresion, '../models/regresion_despacho_model.pkl', compress=3)
joblib.dump(list(X_train.columns), '../models/columnas_regresion.pkl')
joblib.dump(le_bodegas, '../models/encoder_bodegas.pkl') # Importante guardar este también

Entrenando Random Forest (versión optimizada)...
¡Entrenamiento finalizado en segundos!

--- MÉTRICAS DEL MODELO ---
MAE (Error Absoluto Medio): 1.36 días
RMSE (Raíz Error Cuadrático): 1.70 días
R² (Coeficiente de Determinación): 0.2035


['../models/encoder_bodegas.pkl']